# OptiMulti-Video: High-Performance Multimodal Attention (T4 Gpus)

This notebook demonstrates the **OptiMulti-Video** project, featuring:
1. **Custom CUDA Kernel**: Fused Normalization & Projection.
2. **Distributed Training**: FSDP on T4 GPUs.

## 1. Environment Setup

In [1]:
!nvidia-smi

Wed Jan 28 00:25:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the Code
Running this on Colab requires the source code.
**Option A (Recommended)**: Clone your GitHub repository.
**Option B**: Upload the `src/`, `model/`, `training/` folders and `setup.py` manually to the Files tab.

In [2]:
# OPTION A: Clone your repo (Uncommment and replace URL)
!git clone https://github.com/Ferasman979/OptiMulti-Video
# %cd OptiMulti-Video

Cloning into 'OptiMulti-Video'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 17 (delta 0), reused 17 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 10.99 KiB | 2.75 MiB/s, done.


## 3. Compile Custom CUDA Kernel
We use `pip install .` to compile the C++ extension on the attached GPU.

In [3]:
!pip install -v .

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
ERROR: Directory '.' is not installable. Neither 'setup.py' nor 'pyproject.toml' found.


## 4. Run FSDP Distributed Training
We spawn 2 processes (if 2 GPUs are available) to train the model.

In [4]:
!python training/train_fsdp.py

python3: can't open file '/content/training/train_fsdp.py': [Errno 2] No such file or directory


## 5. Verify Custom Kernel
Let's run a quick numerical check to ensure our CUDA kernel matches PyTorch.

In [5]:
import torch
import optimulti_fusion_cuda

if torch.cuda.is_available():
    device = torch.device('cuda')
    a = torch.randn(16, 128, 768, device=device)
    b = torch.randn(16, 128, 768, device=device)
    out_cuda = torch.zeros_like(a)

    # Custom Op
    optimulti_fusion_cuda.fused_add_layernorm(a, b, out_cuda, 1e-5)

    # PyTorch Ref
    out_ref = torch.nn.functional.layer_norm(a + b, (768,), eps=1e-5)

    diff = (out_cuda - out_ref).abs().max().item()
    print(f"Max Difference: {diff}")
    assert diff < 1e-3, "Kernel mismatch!"
    print("verification Passed!")
else:
    print("No GPU available for verification.")

ModuleNotFoundError: No module named 'optimulti_fusion_cuda'